# Understanding `MGnifier`s

Each MGnify API Endpoint has its own MGnifier derivative

---
## Intro to the MGnify RESTful API

### The "resources" vs. "endpoints"?
- In REST (REpresentational State Transfer) styling, data are modelled as **"resources"** which can either be a singleton (e.g., `study`) or collection (collection of singletons e.g. `studies`) resource. More on RESTful APIs [here](https://restfulapi.net/resource-naming/). 

- As explained in its [docs](https://www.ebi.ac.uk/metagenomics/api/v2/): In MGnigy API v2, collection resources are accessed via **"list" endpoints** e.g.
    - `https://www.ebi.ac.uk/metagenomics/api/v2/studies/` or `.../analyses/`
    - `.../studies/<studyID>/analyses/` (a "sub-collection")
    - `.../samples/<sampleID>/runs/` (another sub-collection)

- and singleton resources via **"detail" endpoints** e.g. `https://www.ebi.ac.uk/metagenomics/api/v2/studies/<studyID>` or `.../analyses/<analysisId>`

### Querying a resource
- Many of the list endpoints can be further queried/filtered :) the acceptable query parameters are clearly documented in the [docs](https://www.ebi.ac.uk/metagenomics/api/v2/) again e.g. [`/studies/` example](https://www.ebi.ac.uk/metagenomics/api/v2/#/Studies/list_mgnify_studies)
- typically the query parameters will appear in the url after a `?` as key-value pairs combined via `&`s
- together the resulting http request url would look something like e.g. 
    - [`https://www.ebi.ac.uk/metagenomics/api/v2/studies/?search=tomato&page=1`](https://www.ebi.ac.uk/metagenomics/api/v2/studies/?search=tomato&page=1) 
    - which requests from the `studies` collection resource, the first page of `study` singleton resources with "tomato" in their title 

### Where are the results or MGnify datasets? 
- The resulting datasets such as taxonomic and functional annotation datasets from MGnify pipeline analyses can be downloaded via FTP urls
- These ftp urls are provided in the `downloads` field of detail endpoints such as `/analyses/<analysisId>` and `/studies/<studyId>`

---

## The `proxies` subpackage

Each of the different proxies (e.g., `mgnipy.proxies.StudyDetail`, `mgnipy.proxies.Analyses`) are basically an API endpoint-specific `MGnifier` instance. 

e.g., `mgnipy.MGnipy().studies` is the same as `mgnipy.proxies.Studies()` which is `mgnipy.MGnifier(resource="studies")` plus added functionality that is specific to the studies endpoint!!


### Available proxies in mgnipy

`mgnipy` exposes a set of "proxy" classes that map directly to MGnify API endpoints. Like the 2 RESTful resource types described in [The MGnify RESTful API intro](#the-resources-vs-endpoints), mgnipy has 2 proxy types:

1. **List proxies** (e.g. `Studies`, `Samples`, `Analyses`) which represent collection resources/list endpoints (e.g. `/studies/`, `/samples/`).
2. **Detail proxies** (e.g. `StudyDetail`, `SampleDetail`, `AnalysisDetail`) are used to fetch singleton resources (by accession or id) e.g. `/studies/<studyId>`

These proxies live in the `mgnipy.proxies` subpackage and mirror the API surface documented at https://www.ebi.ac.uk/metagenomics/api/v2/.

#### Brief mapping (proxy → endpoint):
- `Studies` → GET `/studies` (list). See API: https://www.ebi.ac.uk/metagenomics/api/v2/#/Studies/get_mgnify_studies
- `StudyDetail` → GET `/studies/{accession}` (detail). See API: https://www.ebi.ac.uk/metagenomics/api/v2/#/Studies/get_mgnify_study
- `Samples` → GET `/samples` (list). See API: https://www.ebi.ac.uk/metagenomics/api/v2/#/Samples/get_mgnify_samples
- `SampleDetail` → GET `/samples/{accession}` (detail). See API: https://www.ebi.ac.uk/metagenomics/api/v2/#/Samples/get_mgnify_sample
- `Runs` → GET `/runs` and `RunDetail` → `/runs/{accession}`
- `Assemblies` → GET `/assemblies` and `AssemblyDetail` → `/assemblies/{accession}`
- `Analyses` → GET `/analyses` and `AnalysisDetail` → `/analyses/{accession}`
- `Publications` → GET `/publications` and `PublicationDetail` → `/publications/{pubmed_id}`
- `Genomes` / `Catalogues` → catalogue and genome endpoints (catalogues list, genomes within catalogues)
- `Biomes` → GET `/biomes` and `BiomeDetail` → `/biomes/{biome_lineage}`
- TODO insert table

## Example equivalents

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder. 
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---


### Example 1. A `MGnifyList`

#### starting from `MGnipy` client
✨ Recommended ✨ Using the high-level `mgnipy.MGnipy` client:

In [1]:
from mgnipy import MGnipy

# init client w/o caching
MG = MGnipy(cache_dir="temp_example")

# build query set
studies = MG.studies(search="tomato")

# preview
studies.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies?search=tomato&page=1
https://www.ebi.ac.uk/metagenomics/api/v2/studies?search=tomato&page=2


#### ≈ starting from `proxies` subpackage

In [2]:
from mgnipy.proxies import Studies

# init
studies2 = Studies(config=dict(cache_dir="temp_example"), search="tomato")

# we can see same query set as above
studies2.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies?search=tomato&page=1
https://www.ebi.ac.uk/metagenomics/api/v2/studies?search=tomato&page=2


#### ≈ starting from `MGnifier`

In [3]:
from mgnipy import MGnifier

# init
studies3 = MGnifier(
    resource="studies", config=dict(cache_dir="temp_example"), search="tomato"
)

# we can see same query set as above
studies3.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies?search=tomato&page=1
https://www.ebi.ac.uk/metagenomics/api/v2/studies?search=tomato&page=2



### Example 2. A `MGnifyDetail`

#### starting from `MGnipy` client
✨ Recommended ✨ Using the high-level `mgnipy.MGnipy` client:

In [4]:
# using the MGnipy inited above
study = MG.study("MGYS00010257")
study.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00010257


#### ≈ starting from `proxies` subpackage

In [5]:
from mgnipy.proxies import StudyDetail

# init
study2 = StudyDetail(config=dict(cache_dir="temp_example"), accession="MGYS00010257")

# we can see same query set as above
study2.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00010257


#### ≈ starting from `MGnifier`

In [6]:
# init
study3 = MGnifier(
    resource="study", config=dict(cache_dir="temp_example"), accession="MGYS00010257"
)

# we can see same query set as above
study3.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies/MGYS00010257


From the 2 examples above we demonstrated 

1. `mgnipy.MGnipy().studies` is the exact same as `mgnipy.proxies.Studies()` which is just a `mgnipy.MGnifier(resource="studies")` with added `studies`-specific functions.

2. `mgnipy.MGnipy().study` is the exact same as `mgnipy.proxies.StudyDetail()` which is just a `mgnipy.MGnifier(resource="study")` with added `study`-specific functions.

... 

And this is the same for the other proxies in mgnipy. 

---

> **TLDR; `mgnipy.MGnifier`s as API Resource `proxies`**
> 
> `mgnipy.MGnipy().studies` is the exact same as `mgnipy.proxies.Studies()` 
> 
> which is just a `mgnipy.MGnifier(resource="studies")` with added studies-specific functions.

And this is the same for all of the resource proxies (analyses, analysis, study, samples, etc.) not just "studies" in the above example. 

---